# Custom Baseline: STGCN (Spatio-Temporal Graph Convolutional Network)

This notebook trains and evaluates an alternative baseline deep learning architecture—**STGCN**—for flood early-warning forecasting in Sri Lanka.

## Architectural Contrast: STGCN vs. Heshan's TF-STGNN

| Feature / Stream | Heshan's TF-STGNN Architecture | Custom STGCN Architecture | Benefits / Rationale |
|---|---|---|---|
| **Temporal Feature Extraction** | 2-layer Recurrent Neural Network (GRU) + Temporal Attention | 1D Temporal Convolutions (TCN) with Gated Linear Units (GLU) | Faster parallel GPU computation; avoids vanishing gradients over sequence lookback |
| **Graph Convolution** | Relational GATv2 with dynamic attention & edge attributes | Standard Normalized Spatial Adjacency GCN ($D^{-1/2} \tilde{A} D^{-1/2} X W$) | Spectral graph convolution baseline without complex attention overhead |
| **Terrain / Static Fusion** | FiLM (Feature-wise Linear Modulation: $\gamma \odot h + \beta$) | Direct Static Feature Concatenation & Linear Projection | Simple, direct feature concatenation benchmark |
| **Block Topology** | Multi-stream parallel fusion | Sandwich ST-Conv Blocks (Temporal Conv → Spatial GCN → Temporal Conv) | Standard spatio-temporal deep learning design (Yu et al., IJCAI 2018) |

---

## Setup Instructions on Kaggle

1. **Add Input**: Attach the dataset `uom230429e/sri-lanka-flood-tabular-graph-2003-2025` via the right-hand panel.
2. **Accelerator**: Select **GPU T4 x2** (or P100).
3. **Internet**: Turn **On** (needed to clone the repository).
4. Click **Save & Run All**.

In [ ]:
# --- Step 1: Clone repository on Kaggle -----------------------------------
import shutil, subprocess, sys

REPO = "https://github.com/heshannethmina/Srilanka-Flood-Data-Set-Creation"
BRANCH = "manuja_baseline"
DEST = "/kaggle/working/repo"

shutil.rmtree(DEST, ignore_errors=True)
print(f"Cloning branch '{BRANCH}' from {REPO}...")
subprocess.run(["git", "clone", "-b", BRANCH, "-q", REPO, DEST], check=True)

print("Repository latest commit:")
print(subprocess.run(["git", "-C", DEST, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

!ls -la {DEST}/model/stgcn

In [ ]:
# --- Step 2: Environment & Dataset Sanity Check -------------------------
import glob, os, torch

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — Enable GPU Accelerator in settings'
print(f"PyTorch Version: {torch.__version__} | Accelerator: {device_name}")

hits = glob.glob("/kaggle/input/**/flood_dataset.parquet", recursive=True)
print(f"Dataset check: {'OK -> ' + hits[0] if hits else 'MISSING — Attach tabular dataset via Add Input'}")
assert hits, "Please attach 'uom230429e/sri-lanka-flood-tabular-graph-2003-2025' to the notebook inputs."

In [ ]:
# --- Step 3: Run STGCN Baseline Training & Evaluation ---------------------
# Evaluates STGCN across Temporal split, Basin Holdout, and Ensemble modes
!python /kaggle/working/repo/model/stgcn_kaggle_run.py --stage all --epochs 60

In [ ]:
# --- Step 4: Package Results ----------------------------------------------
import glob, json, os

result_files = sorted(glob.glob("/kaggle/working/runs/stgcn_*.json"))
print(f"Found {len(result_files)} STGCN result file(s):")
for f in result_files:
    print(f"  {os.path.basename(f):35s} ({os.path.getsize(f) / 1e3:.1f} kB)")
    with open(f) as fp:
        data = json.load(fp)
        t = data['test']
        print(f"    -> Protocol: {data['protocol']:10s} | PR-AUC: {t['pr_auc']:.4f} | ROC-AUC: {t['roc_auc']:.4f} | Brier: {t['brier']:.5f} | CSI: {t['csi']:.3f}")

if result_files:
    !cd /kaggle/working && zip -qr stgcn_runs.zip runs && ls -lh stgcn_runs.zip